# Llama : syntax

In [ ]:
from llama_cloud import LlamaCloud

client = LlamaCloud() 

file = client.files.create(file="/home/rajnikant/Github/filings/data/raw_filings/aapl-20250927.pdf", purpose="parse")
result = client.parsing.parse(
    file_id=file.id,
    tier="agentic",
    version="latest",
    expand=["markdown"],
)


# Llama : url parsing

In [ ]:
import requests
import json
from pathlib import Path
from llama_cloud import LlamaCloud

# Initialize client
client = LlamaCloud()

# Download SEC filing locally
url = "https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm"

headers = {
    "User-Agent": "MyCompanyName myemail@company.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov",
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

local_path = Path("outputs/aapl-20250927.htm")
local_path.parent.mkdir(parents=True, exist_ok=True)
local_path.write_text(response.text, encoding="utf-8")

# Upload to LlamaCloud
file = client.files.create(
    file=str(local_path),
    purpose="parse"
)


# Strict heading prompt
LLAMAPARSE_10K_PROMPT = """

This is a SEC Form 10-K or 10-Q filing. STRICT OUTPUT RULES:

HEADING HIERARCHY:
- Only 'PART I', 'PART II', 'PART III', 'PART IV' use heading level # (H1). Nothing else uses #.
- Only 'Item X.' entries (Item 1., 1A., 1B., 1C., 2., 3., 4., 5., 6., 7., 7A., 8., 9., 9A., 9B., 9C., 10., 11., 12., 13., 14., 15., 16.) use heading level ## (H2). Nothing else uses ##.
- All headings based on visual hierarchy, text size, and style should be of ### (H3). 
- Hierarchy resets per Item.

FORMATTING RULES:
- NEVER USE any headings except H1, H2, H3.
- NEVER use **bold**, *italic*, __underline__, `code`, or any text formatting
- NEVER use HTML tags like <b>, <i>, <strong>, <em>
- NEVER use markdown * emphasis characters
- NEVER use markdown emphasis characters: *, _, `, ~, >, -, +, |
- NEVER use bullet points, numbered lists, tables, blockquotes, or horizontal rules
- Output headings as plain text with # symbols only
- All body text must be plain text without any formatting wrappers
- Preserve exact Item numbering and titles
- Never skip heading levels
- Never use # or ## inside Item content

"""


# Parse with custom prompt
result = client.parsing.parse(
    file_id=file.id,
    tier="agentic",
    version="latest",
    expand=["markdown"],
    agentic_options={
        "custom_prompt": LLAMAPARSE_10K_PROMPT
    }
)

# Save outputs
output_file = Path("outputs/parsed_output_url.md")
output_file.parent.mkdir(parents=True, exist_ok=True)

# Full markdown
full_md = "\n\n".join(page.markdown for page in result.markdown.pages)
output_file.write_text(full_md, encoding="utf-8")

print(full_md)


UNITED STATES
SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

FORM 10-K

(Mark One)

[x] ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the fiscal year ended September 27, 2025

or

[ ] TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from      to     .

Commission File Number: 001-36743

### Apple Inc.

(Exact name of Registrant as specified in its charter)

California 94-2404110

(State or other jurisdiction of incorporation or organization) (I.R.S. Employer Identification No.)

One Apple Park Way
Cupertino, California 95014

(Address of principal executive offices) (Zip Code)

(408) 996-1010

(Registrant’s telephone number, including area code)

Securities registered pursuant to Section 12(b) of the Act:

<table>
  <thead>
    <tr>
        <th>Title of each class</th>
        <th>Trading symbol(s)</th>
        <th>Name of each exchange on which registered</th>
  

# Llama : pdf parsing

In [9]:
import json
from pathlib import Path
from llama_cloud import LlamaCloud

# Initialize client with API key in code
client = LlamaCloud()

# Local PDF file path
pdf_path = Path("/home/rajnikant/Github/filings/data/raw_filings/tsla-20250630-gen.pdf")

# Upload PDF to LlamaCloud
file = client.files.create(
    file=str(pdf_path),
    purpose="parse",
)

# Strict heading prompt
LLAMAPARSE_10K_PROMPT = """

This is a SEC Form 10-K or 10-Q filing. STRICT OUTPUT RULES:

HEADING HIERARCHY:
- Only 'PART I', 'PART II', 'PART III', 'PART IV' use heading level # (H1). Nothing else uses #.
- Only 'Item X.' entries (Item 1., 1A., 1B., 1C., 2., 3., 4., 5., 6., 7., 7A., 8., 9., 9A., 9B., 9C., 10., 11., 12., 13., 14., 15., 16.) use heading level ## (H2). Nothing else uses ##.
- All headings based on visual hierarchy, text size, and style should be of ### (H3). 
- Hierarchy resets per Item.

FORMATTING RULES:
- MUST parse TABLE perfectly 100 percent exact same as its in the page .
- NEVER USE any headings except H1, H2, H3.
- NEVER use **bold**, *italic*, __underline__, `code`, or any text formatting
- NEVER use HTML tags like <b>, <i>, <strong>, <em>
- NEVER use markdown * emphasis characters
- NEVER use markdown emphasis characters: *, _, `, ~, >, -, +, |
- NEVER use bullet points, numbered lists, tables, blockquotes, or horizontal rules
- Output headings as plain text with # symbols only
- All body text must be plain text without any formatting wrappers
- Preserve exact Item numbering and titles
- Never skip heading levels
- Never use # or ## inside Item content

"""

# Parse with custom prompt
result = client.parsing.parse(
    file_id=file.id,
    tier="agentic",
    version="latest",
    expand=["markdown"],
    agentic_options={
        "custom_prompt": LLAMAPARSE_10K_PROMPT
    }
)


# Save outputs
output_file = Path("outputs/parsed_output_pdf.md")
output_file.parent.mkdir(parents=True, exist_ok=True)

# Full markdown
full_md = "\n\n".join(page.markdown for page in result.markdown.pages)
output_file.write_text(full_md, encoding="utf-8")

print(full_md)


### UNITED STATES SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

### FORM 10-Q

(Mark One)

[x] QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the quarterly period ended June 30, 2025

OR

[ ] TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from      to     

Commission File Number: 001-34756

### Tesla, Inc.

(Exact name of registrant as specified in its charter)

Texas 91-2197729

(State or other jurisdiction of incorporation or organization) (I.R.S. Employer Identification No.)

1 Tesla Road Austin, Texas 78725

(Address of principal executive offices) (Zip Code)

### (512) 516-8177

(Registrant’s telephone number, including area code)

### Securities registered pursuant to Section 12(b) of the Act:

<table>
  <thead>
    <tr>
        <th>Title of each class</th>
        <th>Trading Symbol(s)</th>
        <th>Name of each exchange on which registered</th>
  

In [6]:
intro = result.markdown.pages[0].markdown + "\n\n" + result.markdown.pages[1].markdown
print(intro)

from pathlib import Path

intro_file = Path("outputs/intro.txt")
with open(intro_file, 'w') as f:
    f.write(intro)

NameError: name 'result' is not defined

# Clean markdown 

In [ ]:
import re
import json
from pathlib import Path

INPUT_PATH  = "outputs/parsed_output_pdf.md"
OUTPUT_PATH = "outputs/hierachy.json"


def parse_blocks(md_text):
    lines = md_text.splitlines()
    blocks = []
    current = None

    for line in lines:
        m = re.match(r'^(#{1,3})\s+(.*)', line)
        if m:
            if current is not None:
                current["content"] = current["content"].strip()
                blocks.append(current)
            current = {"level": len(m.group(1)), "heading": m.group(2).strip(), "content": ""}
        else:
            if current is not None:
                current["content"] += line + "\n"

    if current is not None:
        current["content"] = current["content"].strip()
        blocks.append(current)

    return blocks


def merge_empty_h3(blocks):
    result = []
    i = 0
    while i < len(blocks):
        block = blocks[i]
        if block["level"] == 3 and not block["content"]:
            merged_heading = block["heading"]
            j = i + 1
            while j < len(blocks):
                nxt = blocks[j]
                if nxt["level"] == 3:
                    merged_heading = merged_heading + " -> " + nxt["heading"]
                    if nxt["content"]:
                        result.append({"level": 3, "heading": merged_heading, "content": nxt["content"]})
                        i = j + 1
                        break
                    j += 1
                else:
                    result.append({"level": 3, "heading": merged_heading, "content": ""})
                    i = j
                    break
            else:
                result.append({"level": 3, "heading": merged_heading, "content": ""})
                i = j
        else:
            result.append(block)
            i += 1

    return result


def build_tree(blocks):
    tree = []
    current_l1 = None
    current_l2 = None

    # Counters: [l1, l2, l3]
    counters = [0, 0, 0]

    for block in blocks:
        title       = re.sub(r'[\s:.\-–—*_|/\\]+$', '', block["heading"]).strip()
        raw         = block["content"]
        has_content = bool(raw)
        has_table   = bool(re.search(r'<table', raw, re.IGNORECASE)) if raw else False
        text        = f"{title}: {raw}" if raw else ""

        if block["level"] == 1:
            counters[0] += 1
            counters[1]  = 0   # reset l2 and l3 when l1 changes
            counters[2]  = 0
            section_no   = f"{counters[0]}"

        elif block["level"] == 2:
            counters[1] += 1
            counters[2]  = 0   # reset l3 when l2 changes
            section_no   = f"{counters[0]}.{counters[1]}"

        elif block["level"] == 3:
            counters[2] += 1
            section_no   = f"{counters[0]}.{counters[1]}.{counters[2]}"

        node = {
            "section_no":  section_no,
            "level":       block["level"],
            "title":       title,
            "has_content": has_content,
            "has_table":   has_table,
            "children":    [],
        }
        if text:
            node["text"] = text

        if block["level"] == 1:
            tree.append(node)
            current_l1 = node
            current_l2 = None

        elif block["level"] == 2:
            if current_l1 is None:
                counters[0] += 1
                current_l1 = {
                    "section_no": f"{counters[0]}",
                    "level": 1, "title": "", "has_content": False,
                    "has_table": False, "children": []
                }
                tree.append(current_l1)
            current_l1["children"].append(node)
            current_l2 = node

        elif block["level"] == 3:
            if current_l2 is None:
                if current_l1 is None:
                    counters[0] += 1
                    current_l1 = {
                        "section_no": f"{counters[0]}",
                        "level": 1, "title": "", "has_content": False,
                        "has_table": False, "children": []
                    }
                    tree.append(current_l1)
                counters[1] += 1
                current_l2 = {
                    "section_no": f"{counters[0]}.{counters[1]}",
                    "level": 2, "title": "", "has_content": False,
                    "has_table": False, "children": []
                }
                current_l1["children"].append(current_l2)
            current_l2["children"].append(node)

    return tree


# --- Run ---
src = Path(INPUT_PATH)
assert src.exists(), f"Input file not found: {INPUT_PATH}"

md_text = src.read_text(encoding="utf-8")
blocks  = parse_blocks(md_text)
blocks  = merge_empty_h3(blocks)
tree    = build_tree(blocks)

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(OUTPUT_PATH).write_text(json.dumps(tree, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Done — {len(tree)} top-level node(s) written to: {OUTPUT_PATH}")

# --- Preview ---
lines = json.dumps(tree, indent=2, ensure_ascii=False).splitlines()
print("\n".join(lines[:60]))
if len(lines) > 60:
    print(f"\n... ({len(lines) - 60} more lines)")

Done — 3 top-level node(s) written to: outputs/hierachy.json
[
  {
    "section_no": "1",
    "level": 1,
    "title": "",
    "has_content": false,
    "has_table": false,
    "children": [
      {
        "section_no": "1.1",
        "level": 2,
        "title": "",
        "has_content": false,
        "has_table": false,
        "children": [
          {
            "section_no": "0.0.1",
            "level": 3,
            "title": "UNITED STATES SECURITIES AND EXCHANGE COMMISSION",
            "has_content": true,
            "has_table": false,
            "children": [],
            "text": "UNITED STATES SECURITIES AND EXCHANGE COMMISSION: Washington, D.C. 20549"
          },
          {
            "section_no": "1.1.2",
            "level": 3,
            "title": "FORM 10-Q",
            "has_content": true,
            "has_table": false,
            "children": [],
            "text": "FORM 10-Q: (Mark One)\n\n[x] QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SE

# Extract: document metadata

In [30]:
from pathlib import Path
from openai import OpenAI
from datetime import date
from pydantic import BaseModel, Field
from typing import Optional, Literal

client = OpenAI()

intro_file = Path("outputs/intro.txt")

with open(intro_file, "r") as f:
    intro = f.read()

class SECFilingDetails(BaseModel):
    company_name: str = Field(
        description="The formal, official name of the corporation or entity filing the report. Output in all CAPS."
    )
    ticker: str = Field(
        description="The official stock exchange trading ticker symbol. Output in all CAPS."
    )
    fiscal_year: int = Field(
        ge=1934,
        le=2030,
        description="The specific four-digit calendar year for which this report is covering."
    )
    filing_type: Literal["10-K", "10-Q", "8-K", "Form 4", "S-1", "Proxy"] = Field(
        description="The standardized regulatory SEC form type or submission designation."
    )
    period_ended: Optional[date] = Field(
        default=None,
        description="The exact balance sheet or reporting period end date (Format: YYYY-MM-DD). If not present, use null."
    )
    # NEW FIELD: Capture the public distribution / submission date
    filing_date: date = Field(
        description="The official date this document was submitted or published to the SEC EDGAR system (Format: YYYY-MM-DD)."
    )

# Structured messages parameter as a list of dicts
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system", 
            "content": "You are an expert financial analyst. Extract SEC filing details accurately. Convert company_name and ticker values to UPPERCASE letters."
        },
        {
            "role": "user", 
            "content": intro
        }
    ],
    temperature=0,
    response_format=SECFilingDetails
)

# FIXED: Added [0] index to resolve the list AttributeError
event_data = response.choices[0].message.parsed

# Print results including the new filing date
if event_data:
    print(f"Company Name: {event_data.company_name}")
    print(f"Ticker:       {event_data.ticker}")
    print(f"Fiscal Year:  {event_data.fiscal_year}")
    print(f"Filing Type:  {event_data.filing_type}")
    print(f"Period Ended: {event_data.period_ended}")
    print(f"Filing Date:  {event_data.filing_date}")
else:
    print("The model refused to parse the input file:", response.choices[0].message.refusal)


Company Name: TESLA, INC.
Ticker:       TSLA
Fiscal Year:  2025
Filing Type:  10-Q
Period Ended: 2025-06-30
Filing Date:  2025-07-17


# Flat Hierachy

In [38]:
import json
from pathlib import Path

INPUT_PATH  = "outputs/hierachy.json"
OUTPUT_PATH = "outputs/hierachy_flat.json"


def flatten_tree(nodes, parent_title=None, result=None):
    if result is None:
        result = []

    for node in nodes:
        flat_node = {
            "section_no":  node.get("section_no", ""),
            "level":       node["level"],
            "title":       node["title"],
            "parent":      parent_title,
            "has_content": node["has_content"],
            "has_table":   node["has_table"],
        }
        if "text" in node:
            flat_node["text"] = node["text"]

        result.append(flat_node)

        if "children" in node:
            flatten_tree(node["children"], parent_title=node["title"], result=result)

    return result


# --- Run ---
src = Path(INPUT_PATH)
assert src.exists(), f"Input file not found: {INPUT_PATH}"

tree = json.loads(src.read_text(encoding="utf-8"))
flat = flatten_tree(tree)

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(OUTPUT_PATH).write_text(json.dumps(flat, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Done — {len(flat)} flat nodes written to: {OUTPUT_PATH}")

# --- Preview ---
lines = json.dumps(flat[:5], indent=2, ensure_ascii=False).splitlines()
print("\n".join(lines))

Done — 100 flat nodes written to: outputs/hierachy_flat.json
[
  {
    "section_no": "1",
    "level": 1,
    "title": "",
    "parent": null,
    "has_content": false,
    "has_table": false
  },
  {
    "section_no": "1.1",
    "level": 2,
    "title": "",
    "parent": "",
    "has_content": false,
    "has_table": false
  },
  {
    "section_no": "0.0.1",
    "level": 3,
    "title": "UNITED STATES SECURITIES AND EXCHANGE COMMISSION",
    "parent": "",
    "has_content": true,
    "has_table": false,
    "text": "UNITED STATES SECURITIES AND EXCHANGE COMMISSION: Washington, D.C. 20549"
  },
  {
    "section_no": "1.1.2",
    "level": 3,
    "title": "FORM 10-Q",
    "parent": "",
    "has_content": true,
    "has_table": false,
    "text": "FORM 10-Q: (Mark One)\n\n[x] QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the quarterly period ended June 30, 2025\n\nOR\n\n[ ] TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES

# content + markdown as only keys

In [46]:
import json
from pathlib import Path
from openai import OpenAI
from datetime import date
from pydantic import BaseModel, Field
from typing import Optional, Literal

FLAT_PATH   = "outputs/hierachy_flat.json"
OUTPUT_PATH = "outputs/final_output.json"

client = OpenAI()

# ── LLM Metadata Schema ───────────────────────────────────────────────────────

class SECFilingDetails(BaseModel):
    company_name: str = Field(
        description="The formal, official name of the corporation or entity filing the report. Output in all CAPS."
    )
    ticker: str = Field(
        description="The official stock exchange trading ticker symbol. Output in all CAPS."
    )
    fiscal_year: int = Field(
        ge=1934, le=2030,
        description="The specific four-digit calendar year for which this report is covering."
    )
    filing_type: Literal["10-K", "10-Q", "8-K", "Form 4", "S-1", "Proxy"] = Field(
        description="The standardized regulatory SEC form type or submission designation."
    )
    period_ended: Optional[date] = Field(
        default=None,
        description="The exact balance sheet or reporting period end date (Format: YYYY-MM-DD). If not present, use null."
    )
    filing_date: date = Field(
        description="The official date this document was submitted or published to the SEC EDGAR system (Format: YYYY-MM-DD)."
    )


# ── LLM Metadata Extraction ───────────────────────────────────────────────────

def extract_metadata(flat_nodes):
    intro_chunks = [n["text"] for n in flat_nodes if n.get("text")][:5]
    intro = "\n\n".join(intro_chunks)

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are an expert financial analyst. Extract SEC filing details accurately. Convert company_name and ticker values to UPPERCASE letters."
            },
            {
                "role": "user",
                "content": intro
            }
        ],
        temperature=0,
        response_format=SECFilingDetails
    )
    parsed = response.choices[0].message.parsed
    return {
        "company_name": parsed.company_name,
        "ticker":       parsed.ticker,
        "fiscal_year":  parsed.fiscal_year,
        "filing_type":  parsed.filing_type,
        "period_ended": str(parsed.period_ended) if parsed.period_ended else None,
        "filing_date":  str(parsed.filing_date),
    }


# ── Build Final Output ────────────────────────────────────────────────────────

import hashlib

def get_file_id(filepath: str) -> str:
    content = Path(filepath).read_bytes()
    return hashlib.md5(content).hexdigest()[:12]   # e.g. "a3f9c2b1e047"

file_id = get_file_id("/home/rajnikant/Github/filings/data/raw_filings/tsla-20250630-gen.pdf")

def build_final(flat_nodes, llm_meta, file_id):
    final = []
    for node in flat_nodes:
        metadata = {
            "file_id":      file_id,          # ← added
            "company_name": llm_meta["company_name"],
            "ticker":       llm_meta["ticker"],
            "fiscal_year":  llm_meta["fiscal_year"],
            "filing_type":  llm_meta["filing_type"],
            "period_ended": llm_meta["period_ended"],
            "filing_date":  llm_meta["filing_date"],
            "section_no":   node.get("section_no", ""),
            "level":        node["level"],
            "title":        node["title"],
            "parent":       node["parent"],
            "has_content":  node["has_content"],
            "has_table":    node["has_table"],
        }
        final.append({
            "content":  node.get("text", ""),
            "metadata": metadata,
        })
    return final


# ── Run ───────────────────────────────────────────────────────────────────────

src = Path(FLAT_PATH)
assert src.exists(), f"Flat file not found: {FLAT_PATH}"

flat = json.loads(src.read_text(encoding="utf-8"))
print(f"Loaded {len(flat)} flat nodes from {FLAT_PATH}")

print("Extracting LLM metadata...")
llm_meta = extract_metadata(flat)
print(f"  {llm_meta}")

final = build_final(flat, llm_meta, file_id)

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(OUTPUT_PATH).write_text(json.dumps(final, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Done — {len(final)} records written to {OUTPUT_PATH}")

# --- Preview first record ---
print("\n--- Preview (record 0) ---")
print(json.dumps(final[0], indent=2, ensure_ascii=False))

Loaded 100 flat nodes from outputs/hierachy_flat.json
Extracting LLM metadata...
  {'company_name': 'TESLA, INC', 'ticker': 'TSLA', 'fiscal_year': 2025, 'filing_type': '10-Q', 'period_ended': '2025-06-30', 'filing_date': '2025-07-17'}
Done — 100 records written to outputs/final_output.json

--- Preview (record 0) ---
{
  "content": "",
  "metadata": {
    "file_id": "d35072b96ed7",
    "company_name": "TESLA, INC",
    "ticker": "TSLA",
    "fiscal_year": 2025,
    "filing_type": "10-Q",
    "period_ended": "2025-06-30",
    "filing_date": "2025-07-17",
    "section_no": "1",
    "level": 1,
    "title": "",
    "parent": null,
    "has_content": false,
    "has_table": false
  }
}
